# Notebook 2 — Regresión: predecir un número
### Redes Neuronales para problemas de salud

<a href="https://colab.research.google.com/github/calderonf/NeuralNetworkSummer/blob/main/notebooks/02_Regresion_Prediccion_Numerica_en_Salud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

**AI Workshop: Neural Networks** — *AgeTech Latam Summer School 2026*  
Eng. Elec. Francisco Calderón — Profesor Asistente, Departamento de Electrónica, PUJ

---

## Clasificación vs. Regresión

| | Pregunta que responde | Ejemplo del taller |
|---|---|---|
| **Clasificación** (Notebook 1) | ¿A qué grupo pertenece? | ¿Tumor benigno o maligno? |
| **Regresión** (este notebook) | ¿Qué número/cantidad? | ¿Cuánto progresará la diabetes en un año? |

La red es casi la misma. Solo cambian **dos cosas**: la neurona de salida (ya no da una clase, da un número) y la forma de medir el error. Vamos a verlo.


## 0. Preparación


In [ ]:
!pip install -q scikit-learn matplotlib numpy pandas tensorflow 2>/dev/null
print('Entorno listo')

## 1. Dataset: progresión de la diabetes

Dataset médico clásico incluido en scikit-learn. Para 442 pacientes tenemos 10 mediciones (edad, IMC, presión arterial, análisis de sangre...) y un **número** que indica la **progresión de la enfermedad un año después**. Queremos predecir ese número.


In [ ]:
from sklearn.datasets import load_diabetes
import pandas as pd

datos = load_diabetes()
X, y = datos.data, datos.target

df = pd.DataFrame(X, columns=datos.feature_names)
df['progresion'] = y
print(f'{X.shape[0]} pacientes, {X.shape[1]} mediciones cada uno')
print(f'La progresion (lo que queremos predecir) va de {y.min():.0f} a {y.max():.0f}')
df.head()

In [ ]:
import matplotlib.pyplot as plt

# El IMC (indice de masa corporal) suele relacionarse con la progresion. Veamoslo:
plt.figure(figsize=(6,4))
plt.scatter(df['bmi'], df['progresion'], alpha=0.5)
plt.xlabel('IMC (normalizado)'); plt.ylabel('Progresion de la diabetes')
plt.title('Cada punto es un paciente')
plt.show()

## 2. Separar y normalizar


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
escala = StandardScaler().fit(X_tr)
X_tr = escala.transform(X_tr)
X_te = escala.transform(X_te)
print('Listo:', len(X_tr), 'para entrenar,', len(X_te), 'para probar')

## 3. La red de regresión

Usamos `MLPRegressor` (antes usábamos `MLPClassifier`). La estructura de capas es idéntica; el `Regressor` ya sabe que la salida es un número.


In [ ]:
from sklearn.neural_network import MLPRegressor

red = MLPRegressor(
    hidden_layer_sizes=(32, 16),   # experimenta con este tamano
    activation='relu',
    alpha=0.1,                     # regularizacion: evita que la red memorice
    learning_rate_init=0.001,
    max_iter=3000,
    early_stopping=True,           # se detiene cuando deja de mejorar
    n_iter_no_change=30,
    random_state=0
)
red.fit(X_tr, y_tr)
print('Red entrenada')

## 4. ¿Qué tan buenas son las predicciones?

En regresión no hablamos de 'precisión %'. Usamos:
- **MAE** (error absoluto medio): en promedio, ¿por cuánto nos equivocamos?
- **R²**: qué tan bien explica el modelo (1.0 = perfecto, 0 = no explica nada).


In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

pred = red.predict(X_te)
print(f'MAE (error promedio): {mean_absolute_error(y_te, pred):.1f} unidades de progresion')
print(f'R2:                   {r2_score(y_te, pred):.2f}')
print()
print('Nota: este dataset es dificil de verdad. Un R2 cercano a 0.3-0.5 es lo normal,')
print('incluso para los mejores modelos. No todo problema real se predice al 99%.')

### Predicho vs. real

Si el modelo fuera perfecto, todos los puntos caerían sobre la línea diagonal.


In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_te, pred, alpha=0.6)
lims = [y_te.min(), y_te.max()]
plt.plot(lims, lims, 'r--', label='prediccion perfecta')
plt.xlabel('Progresion REAL'); plt.ylabel('Progresion PREDICHA')
plt.title('Predicho vs. Real'); plt.legend(); plt.show()

## 5. Predecir un paciente nuevo

Así se usaría el modelo en la práctica: le pasamos las mediciones de un paciente y devuelve una predicción.


In [ ]:
paciente = X_te[0].reshape(1, -1)   # tomamos un paciente de prueba como ejemplo
print(f'Progresion predicha: {red.predict(paciente)[0]:.0f}')
print(f'Progresion real:     {y_te[0]:.0f}')

## 6. Lo mismo con Keras (opcional)

Fíjate en la última capa: `Dense(1)` **sin** función de activación (queremos un número libre, no una probabilidad). Y la pérdida es `mse` (error cuadrático medio) en vez de `binary_crossentropy`. Esas son las únicas diferencias reales frente al Notebook 1.


In [ ]:
import tensorflow as tf
from tensorflow import keras

modelo = keras.Sequential([
    keras.layers.Input(shape=(X_tr.shape[1],)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1)               # salida: un numero (sin activacion)
])
modelo.compile(optimizer='adam', loss='mse', metrics=['mae'])

historia = modelo.fit(X_tr, y_tr, validation_split=0.1,
                      epochs=200, batch_size=16, verbose=0)

loss, mae = modelo.evaluate(X_te, y_te, verbose=0)
print(f'MAE en prueba (Keras): {mae:.1f}')

plt.plot(historia.history['loss'], label='entrenamiento')
plt.plot(historia.history['val_loss'], label='validacion')
plt.xlabel('Epoca'); plt.ylabel('Error (MSE)'); plt.legend()
plt.title('Curva de aprendizaje'); plt.show()

## 7. Experimenta

1. Cambia `hidden_layer_sizes` en la sección 3. ¿Mejora el R²? ¿O empeora (sobreajuste)?
2. En Keras, sube o baja las `epochs`. Observa cuándo la curva de validación deja de bajar.
3. Reto: usa una sola característica (`X[:, [2]]`, el IMC) y compara. ¿Cuánta información aportan las demás mediciones?


---
## Resumen del taller

- **Clasificación** y **regresión** usan la misma red; cambian la salida y la métrica de error.
- El flujo siempre es el mismo: **datos -> separar/normalizar -> definir la red -> `fit` -> evaluar -> `predict`**.
- Con pocas líneas de Python puedes aplicar redes neuronales a problemas reales de salud.
- El [TensorFlow Playground](https://playground.tensorflow.org/) es tu intuición visual; estos notebooks son esa intuición hecha código.

¡Gracias por participar! 🧠
